# Day 04 — NGS pathway delta P trên subset 64 bệnh nhân

**Slide tham khảo chi tiết:** [day04_reference_pack.zip](_static/slides/day04_reference_pack.zip)

## Mục tiêu bài học

- tạo predicted probability p(EGFR+) từ mô hình minh họa
- ghép dữ liệu dự đoán với bảng pathway 64 bệnh nhân
- tính delta mean P và delta median P giữa nhóm mutated và WT
- vẽ bar chart và boxplot cho pathway nổi bật

## Nội dung

Day 04 là cầu nối giữa machine learning và radiogenomics. Ở đây, predicted probability không còn chỉ là một con số mô hình, mà trở thành đầu vào để so sánh theo pathway. Trọng tâm là đọc kết quả thận trọng, không diễn giải quá mức.

## Sản phẩm sau bài học

- `pathway_delta_p_summary.csv`
- `pathway_delta_median_bar.png`
- `top_pathway_boxplot.png`
- ghi chú ngắn về pathway có chênh lệch lớn nhất


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
plt.rcParams["figure.figsize"] = (6,4)
sns.set_style("whitegrid")

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

GITHUB_USER = "ketnoimaytinh797-dotcom"
REPO_NAME = "EGFR-Radiomics-MiniBootcamp"
BRANCH = "main"


def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out = Path(rel_path).name
    urllib.request.urlretrieve(url, out)
    return Path(out)


def load_csv(rel_path: str) -> pd.DataFrame:
    p = Path(rel_path)
    if p.exists():
        return pd.read_csv(p)
    try:
        p2 = download_from_github(rel_path)
        return pd.read_csv(p2)
    except Exception as e:
        raise FileNotFoundError(f"Không tìm thấy {rel_path}. Hãy chỉnh đúng GITHUB_USER/REPO_NAME hoặc upload file thủ công.") from e


df = load_csv("data/nsclc_egfr_radiomics_simplified.csv")
ngs = load_csv("data/ngs_pathway_demo_64.csv")

## 1. Train mô hình minh họa và lấy predicted probability

In [ ]:

feature_set = "ring1"
cols = [c for c in df.columns if c.startswith(feature_set + "_")]
X = df[cols]
y = df["egfr_mutation"].astype(int)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])
pipe.fit(X, y)
df["pred_prob_egfr"] = pipe.predict_proba(X)[:, 1]

df[["patient_id", "pred_prob_egfr"]].head()


## 2. Ghép với bảng pathway 64 bệnh nhân

In [ ]:

merged = ngs.merge(df[["patient_id", "pred_prob_egfr"]], on="patient_id", how="left")
merged.head(), merged.shape


## 3. Tính delta mean P và delta median P theo từng pathway

In [ ]:

pathway_cols = [c for c in merged.columns if c.startswith("pathway_")]
rows = []
for pw in pathway_cols:
    g_mut = merged.loc[merged[pw] == 1, "pred_prob_egfr"].dropna()
    g_wt = merged.loc[merged[pw] == 0, "pred_prob_egfr"].dropna()
    delta_mean = g_mut.mean() - g_wt.mean()
    delta_median = g_mut.median() - g_wt.median()
    p = mannwhitneyu(g_mut, g_wt, alternative="two-sided").pvalue
    rows.append([
        pw.replace("pathway_", ""),
        len(g_mut), len(g_wt),
        g_mut.mean(), g_wt.mean(),
        delta_mean,
        g_mut.median(), g_wt.median(),
        delta_median,
        p
    ])

pathway_summary = pd.DataFrame(rows, columns=[
    "pathway", "n_mut", "n_wt",
    "mean_p_mut", "mean_p_wt", "delta_mean_p",
    "median_p_mut", "median_p_wt", "delta_median_p",
    "p_value"
]).sort_values("delta_median_p", ascending=False)
pathway_summary


## 4. Vẽ bar chart theo delta median P

In [ ]:

out_dir = Path("outputs/day04")
out_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(7,4))
pathway_summary.plot(x="pathway", y="delta_median_p", kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Delta median P")
ax.set_title("Pathway-level delta median predicted probability")
fig.tight_layout()
fig.savefig(out_dir / "delta_median_p_bar.png", dpi=150)
plt.show()


## 5. Boxplot cho một pathway cụ thể

Ví dụ chọn pathway có delta median P lớn nhất để minh họa.

In [ ]:

top_pathway = pathway_summary.iloc[0]["pathway"]
pw_col = f"pathway_{top_pathway}"
plot_df = merged[[pw_col, "pred_prob_egfr"]].copy()
plot_df["group"] = plot_df[pw_col].map({0: "WT", 1: "Mutated"})

fig, ax = plt.subplots()
sns.boxplot(data=plot_df, x="group", y="pred_prob_egfr", ax=ax)
ax.set_title(f"Predicted probability by {top_pathway} pathway")
fig.tight_layout()
fig.savefig(out_dir / f"boxplot_{top_pathway}.png", dpi=150)
plt.show()


## 6. Cách diễn giải kết quả

Khi nhìn Day 04, nên đi theo 3 câu hỏi:

1. Pathway nào có delta median P cao hoặc thấp đáng chú ý?  
2. Chênh lệch này có đi cùng p value nhỏ hay chỉ là tín hiệu gợi ý?  
3. Tín hiệu đó có phù hợp với câu chuyện sinh học của dự án hay không?

Mục tiêu ở đây là diễn giải thận trọng. Không nên nói pathway gây ra EGFR, mà chỉ nên nói mô hình dự đoán cho thấy sự khác biệt xác suất theo nhóm pathway trong subset minh họa.

In [ ]:

pathway_summary.to_csv(out_dir / "pathway_delta_p_summary.csv", index=False)
pathway_summary


## 7. Tự kiểm tra

1. Vì sao Day 04 dùng thêm median bên cạnh mean?  
2. Delta mean P và delta median P khác nhau ở điểm nào?  
3. Nếu pathway có n rất nhỏ thì phải cẩn thận điều gì?  
4. Tại sao Day 04 chỉ nên diễn giải ở mức gợi ý sinh học?

## Nối sang buổi sau

Day 05 sẽ ghép lại toàn bộ flow của 4 buổi trước để tạo ra một notebook capstone và thư mục output hoàn chỉnh.